# CIFAR-10-LT · PLWCE α-sweep 파일럿

`plwce`의 최적 지수 α\*가 **불균형비(IR)와 어떤 함수 관계**인지 확인하기 위한 분석용 노트북.
메인 실험(`CIFAR10_LT.ipynb`)과 동일한 데이터·모델·학습 설정을 재사용하되, **plwce 한 종만**
α를 촘촘히 sweep 한다. Optuna proxy(20ep·subset)가 아니라 **full-ish epochs**로 돌려 α\* 추정을 안정화.

**설계**
- IR ∈ {10, 20, 50, 100, 200} — 같은 데이터셋에서 IR만 통제변수로 변화
- α ∈ {0.5, 1.0, …, 6.0} (0.5 간격, 12점)
- seeds = {42,43,44} (파일럿 3개), `SWEEP_EPOCHS=100`
- α\* = (seed 평균 test F1-Macro) 곡선의 정점 (top-3 2차 피팅으로 연속 보정)
- α\* vs **IR** 과 vs **log(IR)** 를 각각 1차 피팅 → R² 비교로 함수형 식별

> ⚠️ **비용**: 기본값 5 IR × 12 α × 3 seed × 100ep = **180 full-ish runs**. T4 기준 대략 20~35 GPU-h.
> 먼저 곡선 모양만 보려면 Cell 1의 `QUICK=True` (2 IR × 6 α × 1 seed × 60ep = 12 runs)로 스모크 테스트.
> per-run 체크포인트라 중단/재개 가능. 최종 논문 수치는 `SWEEP_EPOCHS=200`(메인과 동일) 권장.

In [1]:
# === Cell 0: 환경 설정 ===
!pip install torchvision pandas -q

import os, sys, json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from sklearn.metrics import f1_score

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/imbalanced-data-LWCE'
# custom_losses.py / experiment_utils.py 는 REPO 루트, resnet32.py 는 image_classification/.
sys.path.insert(0, f'{REPO}/image_classification')
sys.path.insert(0, REPO)

from custom_losses import get_clf_loss
from experiment_utils import GradLogger, extended_metrics
from resnet32 import build_resnet32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# ==================================================================
# 실험 설정 (sweep 파라미터)
# ==================================================================
DATASET     = 'cifar10'
NUM_CLASSES = 10

IR_LIST     = [10, 20, 50, 100, 200]                   # IR 통제변수
SEEDS       = [42, 43, 44]
SWEEP_EPOCHS = 100                                      # proxy(20) < 100 < final(200)

# 3차 피드백 §4.6: "α와 ε 변화에 따른 Macro-F1, Worst-class Acc, Gradient Norm"
#   → α(PLWCE) 뿐 아니라 ε(ES-LWCE) sweep도 함께 돈다.
#   ε는 효과가 배수적이라 logspace (ε→0: LWCE 극한 / ε→∞: CE 극한).
ALPHA_GRID  = [round(a, 2) for a in np.arange(0.5, 6.0001, 0.5)]        # 12점
EPS_GRID    = [round(float(e), 4) for e in np.logspace(-1, 1, 12)]      # 12점
SWEEPS = [
    ('plwce',  'alpha', ALPHA_GRID),
    ('eslwce', 'eps',   EPS_GRID),
]

BATCH_SIZE  = 128
NUM_WORKERS = 0
LR          = 0.1
SEED        = 42

# --- 빠른 스모크 테스트 토글 ---
QUICK = False
if QUICK:
    IR_LIST      = [10, 100]
    SWEEPS       = [('plwce', 'alpha', [0.5, 2.5, 4.5]), ('eslwce', 'eps', [0.1, 1.0, 10.0])]
    SEEDS        = [42]
    SWEEP_EPOCHS = 60

RESULTS_BASE = f'{REPO}/image_classification/results/CIFAR10_alpha_sweep'
os.makedirs(RESULTS_BASE, exist_ok=True)
CKPT = f'{RESULTS_BASE}/alpha_sweep_checkpoint.json'

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)

n_runs = len(IR_LIST) * sum(len(g) for _, _, g in SWEEPS) * len(SEEDS)
print('\n설정 완료')
print(f'  IR={IR_LIST}  seeds={SEEDS}  epochs={SWEEP_EPOCHS}')
for ln, prm, g in SWEEPS:
    print(f'  {ln:7s} {prm:5s} grid({len(g)}) = {g}')
print(f'  총 {n_runs} runs -> 체크포인트: {CKPT}')


Mounted at /content/drive
Device: cuda | NVIDIA L4

설정 완료
  IR=[10, 20, 50, 100, 200]  seeds=[42, 43, 44]  epochs=100
  plwce   alpha grid(12) = [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.5), np.float64(6.0)]
  eslwce  eps   grid(12) = [0.1, 0.152, 0.231, 0.3511, 0.5337, 0.8111, 1.2328, 1.8738, 2.848, 4.3288, 6.5793, 10.0]
  총 360 runs -> 체크포인트: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_alpha_sweep/alpha_sweep_checkpoint.json


In [2]:
# === Cell 1: CIFAR-LT 데이터 로더 (메인 노트북과 동일 로직) ===

def make_cifar_lt(dataset_name: str, imbalance_ratio: int, seed: int = 42):
    """지수 감소 long-tail: n_i = n_max x IR^(-i/(K-1)). (indices, class_counts) 반환."""
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    if dataset_name == 'cifar10':
        dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    else:
        dataset = datasets.CIFAR100(root='/tmp/cifar', train=True, download=True, transform=None)
    targets = np.array(dataset.targets)
    class_indices = [np.where(targets == c)[0] for c in range(K)]
    rho = imbalance_ratio ** (-1 / (K - 1))
    class_counts = [int(n_max * (rho ** i)) for i in range(K)]
    np.random.seed(seed)
    lt_indices = []
    for c, n_samples in enumerate(class_counts):
        n_samples = max(1, n_samples)
        selected = np.random.choice(class_indices[c], size=n_samples, replace=False)
        lt_indices.extend(selected)
    lt_indices = np.array(lt_indices)
    np.random.shuffle(lt_indices)
    return lt_indices.tolist(), class_counts


def load_cifar_lt_loaders(ir: int, batch_size: int = 128, num_workers: int = 0):
    """CIFAR-10 LT train(80) / val(20) + 표준 test. 메인 노트북과 동일."""
    full_dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    lt_indices, class_counts = make_cifar_lt('cifar10', ir, seed=SEED)
    lt_indices = np.array(lt_indices)
    lt_targets = np.array(full_dataset.targets)[lt_indices]

    train_indices, val_indices = [], []
    for c in range(10):
        c_idx = np.where(lt_targets == c)[0]
        np.random.seed(SEED); np.random.shuffle(c_idx)
        n_c_val = max(1, len(c_idx) // 5)
        val_indices.extend(lt_indices[c_idx[:n_c_val]])
        train_indices.extend(lt_indices[c_idx[n_c_val:]])

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010]),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010]),
    ])
    train_ds = Subset(full_dataset, train_indices); train_ds.dataset.transform = train_tf
    val_ds   = Subset(full_dataset, val_indices)
    test_ds  = datasets.CIFAR10(root='/tmp/cifar', train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader, class_counts


# IR별 로더는 한 번만 만들어 캐시 (alpha/seed 루프에서 재사용)
_loader_cache = {}
def get_loaders(ir):
    if ir not in _loader_cache:
        _loader_cache[ir] = load_cifar_lt_loaders(ir, BATCH_SIZE, NUM_WORKERS)
    return _loader_cache[ir]

print('데이터 로더 정의 완료')

데이터 로더 정의 완료


In [ ]:
# === Cell 2: 모델 / 평가 / 학습 함수 (메인 노트북과 동일) ===
# 3차 피드백 §4.6: sweep에서 F1뿐 아니라 Worst-class Acc / Gradient Norm도 봐야 한다.
#   Worst-class는 per-class recall에서 사후 계산되지만, Gradient Norm은 학습 중에만 잡힌다.

def compute_test_metrics(model, loader, num_classes, class_counts_train):
    """Test F1-Macro / G-Mean / Worst-class / Balanced / Many·Medium·Few (tertile)."""
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            y_pred.extend(model(imgs.to(device)).argmax(dim=1).cpu().tolist())
            y_true.extend(labels.tolist())
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    out = {
        'Top1_Acc': float((y_true == y_pred).mean()),
        'F1_Macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }
    # Per_Class_Acc / G_Mean / Worst_Acc / Balanced_Acc — 전 도메인 공용 정의
    out.update(extended_metrics(y_true, y_pred, num_classes))

    pca = np.array(out['Per_Class_Acc'])
    order = np.argsort(np.array(class_counts_train))[::-1]
    n = len(order)
    out['Many_Acc']   = float(pca[order[:n // 3]].mean())
    out['Medium_Acc'] = float(pca[order[n // 3:2 * n // 3]].mean())
    out['Few_Acc']    = float(pca[order[2 * n // 3:]].mean())
    return out


def compute_val_f1(model, loader):
    model.eval(); all_p, all_l = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            all_p.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            all_l.extend(labels.tolist())
    return f1_score(all_l, all_p, average='macro', zero_division=0)


def train_one(loss_name, params, class_counts, train_loader, val_loader, epochs, seed):
    """손실 1개 학습 -> val F1 최고 모델 + gradient history 반환.
    params: {'alpha': v} 또는 {'eps': v}. 스케줄러는 epochs에 맞춰 스케일."""
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    model = build_resnet32(NUM_CLASSES).to(device)
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=2e-4)
    milestones = [int(epochs * 0.8), int(epochs * 0.9)]          # 200ep->[160,180]과 동일 비율
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.01)
    criterion = get_clf_loss(loss_name, class_counts, **params)
    glog = GradLogger(class_counts, NUM_CLASSES, device)

    best_f1, best_state = 0.0, None
    history = {'train_loss': [], 'val_f1': [],
               'grad_norm': [], 'grad_many': [], 'grad_few': [], 'grad_ratio': []}
    ptxt = ', '.join(f'{k}={v:.3f}' for k, v in params.items())
    pbar = tqdm(range(epochs), desc=f'{loss_name} {ptxt} s{seed}', leave=False)

    for _ in pbar:
        model.train()
        glog.reset()
        run_loss, n_seen = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            logits.retain_grad()                 # 샘플별 로짓 기울기 관측 (Proposition 4)
            loss = criterion(logits, labels)
            loss.backward()
            glog.update(logits, labels, model)   # optimizer.step() 전에 호출
            optimizer.step()
            run_loss += float(loss) * labels.size(0); n_seen += labels.size(0)

        vf1 = compute_val_f1(model, val_loader)
        history['train_loss'].append(run_loss / max(1, n_seen))
        history['val_f1'].append(vf1)
        for k, v in glog.epoch_end().items():
            history[k].append(v)

        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        scheduler.step(); pbar.update()

    if best_state:
        model.load_state_dict(best_state)
    return model, best_f1, history

print('학습/평가 함수 정의 완료 (F1 / G-Mean / Worst-class + gradient 계측)')


학습/평가 함수 정의 완료 (F1 / G-Mean / Worst-class + gradient 계측)


In [4]:
# === Cell 3: sweep 실행 (per-run 체크포인트 재개) ===
# α(PLWCE)와 ε(ES-LWCE) 두 sweep을 같은 루프로 처리한다.
os.environ['TQDM_DISABLE'] = '0'

ckpt = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
print(f'체크포인트: {len(ckpt)}/{n_runs} 완료')

for ir in IR_LIST:
    train_loader, val_loader, test_loader, class_counts = get_loaders(ir)
    for loss_name, param, grid in SWEEPS:
        for val in grid:
            for seed in SEEDS:
                key = f'IR{ir}_{loss_name}_{param}{val:.4f}_s{seed}'
                if key in ckpt:
                    continue
                model, val_f1, hist = train_one(loss_name, {param: val}, class_counts,
                                                train_loader, val_loader,
                                                SWEEP_EPOCHS, seed)
                m = compute_test_metrics(model, test_loader, NUM_CLASSES, class_counts)
                ckpt[key] = {'ir': ir, 'loss': loss_name, 'param': param, 'value': float(val),
                             'seed': seed, 'val_f1': val_f1, 'metrics': m,
                             # §4.6: Gradient Norm은 사후 복구 불가 → 마지막 epoch 값 보존
                             'grad_norm_final': hist['grad_norm'][-1],
                             'grad_ratio_final': hist['grad_ratio'][-1],
                             'history': hist}
                with open(CKPT, 'w') as f:
                    json.dump(ckpt, f)
                print(f'  {key}  testF1={m["F1_Macro"]:.4f}  Worst={m["Worst_Acc"]:.4f}  '
                      f'gradNorm={hist["grad_norm"][-1]:.3f}  ratio={hist["grad_ratio"][-1]:.2f}')
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

print('\nsweep 완료')


체크포인트: 71/360 완료


100%|██████████| 170M/170M [1:03:44<00:00, 44.6kB/s] 


eslwce eps=10.000 s44:   0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_582/910172016.py:70: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run_loss += float(loss) * labels.size(0); n_seen += labels.size(0)


  IR10_eslwce_eps10.0000_s44  testF1=0.8340  Worst=0.7320  gradNorm=0.638  ratio=2.06


...

  IR20_eslwce_eps2.8480_s42  testF1=0.7995  Worst=0.6920  gradNorm=0.645  ratio=2.37


eslwce eps=2.848 s43:   0%|          | 0/100 [00:00<?, ?it/s]

  IR20_eslwce_eps2.8480_s43  testF1=0.7960  Worst=0.6930  gradNorm=0.660  ratio=2.67


eslwce eps=2.848 s44:   0%|          | 0/100 [00:00<?, ?it/s]

: 

In [ ]:
# === Cell 4: 집계 · alpha*(IR) 추정 · 함수형 피팅 · §4.6 시각화 ===
import collections

ckpt = json.load(open(CKPT))

def rows_for(loss_name, param):
    acc = collections.defaultdict(lambda: collections.defaultdict(list))
    for v in ckpt.values():
        if v.get('loss') != loss_name or v.get('param') != param:
            continue
        k = (v['ir'], v['value'])
        acc[k]['F1_Macro'].append(v['metrics']['F1_Macro'])
        acc[k]['Worst_Acc'].append(v['metrics']['Worst_Acc'])
        acc[k]['G_Mean'].append(v['metrics']['G_Mean'])
        acc[k]['grad_norm'].append(v['grad_norm_final'])
        acc[k]['grad_ratio'].append(v['grad_ratio_final'])
    return acc

def refine_peak(xs, means):
    """argmax 인덱스 주변 3점 2차 피팅으로 연속 최적값 보정 (오목할 때만)."""
    i = int(np.nanargmax(means))
    xstar = xs[i]
    if 0 < i < len(xs) - 1 and np.all(np.isfinite(means[i-1:i+2])):
        x = np.array(xs[i-1:i+2]); y = np.array(means[i-1:i+2])
        a2, b2, _ = np.polyfit(x, y, 2)
        if a2 < 0:
            vx = -b2 / (2 * a2)
            if x[0] <= vx <= x[-1]:
                xstar = float(vx)
    return xstar, i

# ── §4.6 그림: 각 sweep마다 (F1 / Worst-class / Gradient Norm) × IR ──
summary = {}
for loss_name, param, grid in SWEEPS:
    acc = rows_for(loss_name, param)
    if not acc:
        print(f'[{loss_name}] 결과 없음 — 스킵'); continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
    cmap = plt.cm.viridis(np.linspace(0, 1, len(IR_LIST)))
    curves, xstar = {}, {}
    for ir, c in zip(IR_LIST, cmap):
        f1m = np.array([np.mean(acc[(ir, v)]['F1_Macro']) if acc[(ir, v)]['F1_Macro'] else np.nan
                        for v in grid])
        f1s = np.array([np.std(acc[(ir, v)]['F1_Macro']) if acc[(ir, v)]['F1_Macro'] else 0.0
                        for v in grid])
        wo  = np.array([np.mean(acc[(ir, v)]['Worst_Acc']) if acc[(ir, v)]['Worst_Acc'] else np.nan
                        for v in grid])
        gn  = np.array([np.mean(acc[(ir, v)]['grad_norm']) if acc[(ir, v)]['grad_norm'] else np.nan
                        for v in grid])
        curves[ir] = {'F1': f1m.tolist(), 'Worst': wo.tolist(), 'grad_norm': gn.tolist()}
        xstar[ir], _ = refine_peak(grid, f1m)

        axes[0].errorbar(grid, f1m, yerr=f1s, marker='o', ms=4, capsize=2, color=c, label=f'IR={ir}')
        axes[0].axvline(xstar[ir], color=c, ls='--', alpha=0.4)
        axes[1].plot(grid, wo, marker='s', ms=4, color=c, label=f'IR={ir}')
        axes[2].plot(grid, gn, marker='^', ms=4, color=c, label=f'IR={ir}')

    for ax, ylab, title in zip(axes,
                               ['Test F1-Macro', 'Worst-class Acc', 'Gradient Norm (final epoch)'],
                               ['F1-Macro', 'Worst-class Accuracy', 'Gradient Norm']):
        ax.set_xlabel(f'{loss_name} {param}'); ax.set_ylabel(ylab)
        ax.set_title(f'{title} vs {param}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        if param == 'eps':
            ax.set_xscale('log')
    axes[2].set_yscale('log')
    plt.suptitle(f'CIFAR-10-LT · {loss_name} {param}-sweep (3차 피드백 §4.6)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_BASE}/sweep_{loss_name}_{param}.png', dpi=130, bbox_inches='tight')
    plt.show()

    summary[loss_name] = {'param': param, 'grid': list(map(float, grid)),
                          'curves': {int(ir): curves[ir] for ir in IR_LIST},
                          'x_star': {int(ir): float(xstar[ir]) for ir in IR_LIST}}
    print(f'\n[{loss_name}] IR별 최적 {param}:')
    for ir in IR_LIST:
        print(f'  IR={ir:4d}  {param}*={xstar[ir]:.3f}')

# ── alpha*(IR) 법칙 피팅 (PLWCE 전용 — 분석/이론 섹션용) ──
if 'plwce' in summary:
    def fit_r2(x, y):
        s, b = np.polyfit(x, y, 1)
        yhat = s * x + b
        ss_res = np.sum((y - yhat) ** 2); ss_tot = np.sum((y - np.mean(y)) ** 2)
        return s, b, (1 - ss_res / ss_tot if ss_tot > 0 else float('nan'))

    irs = np.array(IR_LIST, float)
    ast = np.array([summary['plwce']['x_star'][ir] for ir in IR_LIST])
    s_lin, b_lin, r2_lin = fit_r2(irs, ast)
    s_log, b_log, r2_log = fit_r2(np.log(irs), ast)
    print(f'\nalpha* ~ {s_lin:.5f}*IR      + {b_lin:.3f}   |  R2(linear) = {r2_lin:.3f}')
    print(f'alpha* ~ {s_log:.3f}*log(IR) + {b_log:.3f}   |  R2(log-IR) = {r2_log:.3f}')
    print('\n주의: 두 피팅의 R2 차이는 작아 통계적으로 동등할 수 있다(n=5).')
    print('      log-IR 채택 근거는 R2가 아니라 PLWCE가 log(n) 가중을 쓴다는 차원적 일관성.')
    print('      또한 이 법칙은 CIFAR IR 범위(10~200) 한정 — 외삽 금지.')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].scatter(irs, ast, s=60, zorder=3)
    xx = np.linspace(irs.min(), irs.max(), 100)
    axes[0].plot(xx, s_lin * xx + b_lin, 'r-', label=f'R2={r2_lin:.3f}')
    axes[0].set_xlabel('IR'); axes[0].set_ylabel('alpha*'); axes[0].set_title('alpha* vs IR (raw)')
    axes[1].scatter(np.log(irs), ast, s=60, zorder=3)
    xx2 = np.linspace(np.log(irs).min(), np.log(irs).max(), 100)
    axes[1].plot(xx2, s_log * xx2 + b_log, 'r-', label=f'R2={r2_log:.3f}')
    axes[1].set_xlabel('log(IR)'); axes[1].set_ylabel('alpha*'); axes[1].set_title('alpha* vs log(IR)')
    for ir, a in zip(irs, ast):
        axes[0].annotate(f'{int(ir)}', (ir, a), textcoords='offset points', xytext=(5, 5), fontsize=8)
    for ax in axes:
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'{RESULTS_BASE}/alpha_star_fit.png', dpi=130); plt.show()

    summary['plwce']['fit'] = {'linear': [s_lin, b_lin, r2_lin], 'log': [s_log, b_log, r2_log]}
    pd.DataFrame({'IR': IR_LIST, 'alpha_star': ast}).to_csv(
        f'{RESULTS_BASE}/alpha_star.csv', index=False)

with open(f'{RESULTS_BASE}/sweep_summary.json', 'w') as f:
    json.dump({'IR_LIST': IR_LIST, 'SEEDS': SEEDS, 'SWEEP_EPOCHS': SWEEP_EPOCHS,
               'sweeps': summary}, f, indent=2)
print(f'\n저장: {RESULTS_BASE}/  (sweep_summary.json, alpha_star.csv, sweep_*.png)')
